# OBJECTVES
- Use `surprise` built-in-reader class to process data to work with recommender algorithms
- Use `surprise` to create and cross-validate different recommender algorithms
- Obtain a prediction for a specific user for a particular items

In [ ]:
from surprise import Dataset
from surprise.model_selection import train_test_split

# loading the jokes dataset 
jokes = Dataset.load_builtin(name= "jester")

In [4]:
type(jokes)

surprise.dataset.DatasetAutoFolds

In [5]:
#Split into train and test set
trainset, testset = train_test_split(jokes, test_size= 0.2)

In [6]:
print("Type trainset:", type(trainset), "\n")
print("Type testset :", type(testset))

Type trainset: <class 'surprise.trainset.Trainset'> 

Type testset : <class 'list'>


In [7]:
# checking how large our testset is and what contained in an individual element
print(len(testset))
print(testset[0])

352288
('51759', '16', -2.188)


## Memory-based methods(Neighborhood-based)


In [8]:
from surprise.prediction_algorithms import knns
from surprise.similarities import cosine, msd, pearson
from surprise import accuracy

In [9]:
# calculate the similarity between whichever number is fewer, users or items
print("Number of users: ", trainset.n_users, "\n")
print("Number of items", trainset.n_items, "\n")

Number of users:  58772 

Number of items 140 



Since we have fewer items than users, it will be more efficient to calculate item-item rather than user-user similarity

In [10]:
sim_cos = { "name" : "cosine", "user_based": False}

In [11]:
basic = knns.KNNBasic(sim_options=sim_cos)
basic.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [ ]:
basic.sim #looking at similarity metrics of each of the items to one another by using sim attribute

array([[ 1.        ,  0.57950099,  0.5842934 , ...,  0.33370829,
         0.47611382,  0.50127105],
       [ 0.57950099,  1.        ,  0.60196503, ...,  0.22547918,
         0.7430872 ,  0.39554409],
       [ 0.5842934 ,  0.60196503,  1.        , ..., -0.734845  ,
         0.975147  ,  0.29928293],
       ...,
       [ 0.33370829,  0.22547918, -0.734845  , ...,  1.        ,
         0.22265858,  0.50260806],
       [ 0.47611382,  0.7430872 ,  0.975147  , ...,  0.22265858,
         1.        ,  0.60677268],
       [ 0.50127105,  0.39554409,  0.29928293, ...,  0.50260806,
         0.60677268,  1.        ]])

In [13]:
# testing the model to see how best it performed
predictions = basic.test(testset)

In [15]:
print(accuracy.rmse(predictions))

RMSE: 4.2164
4.216361035923457


In [16]:
sim_pearson = {"name": "pearson", "user_based": False}
basic_pearson = knns.KNNBasic(sim_options=sim_pearson)
basic_pearson.fit(trainset)
predictions = basic_pearson.test(testset)
print(accuracy.rmse(predictions))

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 4.2773
4.277346630374865


In [17]:
sim_pearson = {"name": "pearson", "user_based": False}
knn_means = knns.KNNWithMeans(sim_options=sim_pearson)
knn_means.fit(trainset)
predictions = knn_means.test(testset)
print(accuracy.rmse(predictions))

Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 4.1423
4.1423442964896156


In [19]:
sim_pearson = {"name" : "pearson", "user_based": False}
knn_baseline = knns.KNNBaseline(sim_options = sim_pearson)
knn_baseline.fit(trainset)
predictions = knn_baseline.test(testset)
print(accuracy.rmse(predictions))

Estimating biases using als...
Computing the pearson similarity matrix...
Done computing similarity matrix.
RMSE: 4.1379
4.137854402203418


In [20]:
from surprise.prediction_algorithms import SVD
from surprise.model_selection import GridSearchCV

param_grid = {'n_factors': [20, 100], "n_epochs" : [5,10], 'lr_all': [0.002,0.005],
              'reg_all': [0.4,0.6]}
gs_model = GridSearchCV(SVD, param_grid=param_grid, n_jobs = -1, joblib_verbose=5)
gs_model.fit(jokes)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:   23.2s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:  5.0min
[Parallel(n_jobs=-1)]: Done  80 out of  80 | elapsed:  8.6min finished
